In [1]:
!pip install pandas scikit-learn openai


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [4]:
import os
import re
import json
import time
import random
import pandas as pd
from getpass import getpass
from openai import OpenAI

# =========================================================
# 1) PATHS
# =========================================================
data_dir = "/Users/shinekhantaung/Desktop/GPT-NER/5fold_train_test"
output_dir = os.path.join("/Users/shinekhantaung/Desktop/GPT-NER/Random-Retrieval/random_retrieval_results")
os.makedirs(output_dir, exist_ok=True)

# =========================================================
# 2) API KEY (secure input)
# =========================================================
api_key = getpass("Enter your OpenAI API key: ")
client = OpenAI(api_key=api_key)

# =========================================================
# 3) SETTINGS
# =========================================================
MODEL_NAME = "gpt-4.1"

# column names: dataset နဲ့မကိုက်ရင် ပြင်ပါ
sentence_col = "sentence"
name_col = "gold_names"

# paper style random retrieval k values
k_values = [4, 8, 16, 32]

# reproducibility
RANDOM_SEED = 42

# =========================================================
# 4) HELPERS
# =========================================================
def normalize_text(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s).strip()
    s = re.sub(r"\s+", " ", s)
    return s

def normalize_entity(s: str) -> str:
    s = normalize_text(s)
    return s.casefold()

def parse_gold_names(cell_value):
    """
    Gold name column က
    - blank / NaN
    - one name
    - multiple names separated by ; , |
    ဆိုတာတွေ handle လုပ်မယ်
    """
    if pd.isna(cell_value):
        return []

    text = str(cell_value).strip()
    if not text:
        return []

    parts = re.split(r"[;|,]", text)
    names = [normalize_text(x) for x in parts if normalize_text(x)]

    # deduplicate
    seen = set()
    result = []
    for n in names:
        key = normalize_entity(n)
        if key not in seen:
            seen.add(key)
            result.append(n)
    return result

def extract_marked_names(output_text: str):
    """
    @@name## pattern တွေ extract
    """
    if not isinstance(output_text, str):
        return []

    matches = re.findall(r"@@(.*?)##", output_text, flags=re.DOTALL)
    names = [normalize_text(m) for m in matches if normalize_text(m)]

    # deduplicate
    seen = set()
    result = []
    for n in names:
        key = normalize_entity(n)
        if key not in seen:
            seen.add(key)
            result.append(n)
    return result

def mark_gold_names(sentence: str, gold_names):
    """
    gold names ကို @@ and ## နဲ့ mark လုပ်ပေးမယ်
    longest first replace လုပ်မယ်
    """
    sentence = normalize_text(sentence)

    if not gold_names:
        return sentence

    marked = sentence
    gold_names_sorted = sorted(gold_names, key=len, reverse=True)

    for name in gold_names_sorted:
        name = normalize_text(name)
        if not name:
            continue

        pattern = re.escape(name)
        marked = re.sub(pattern, f"@@{name}##", marked)

    return marked

def build_prompt(test_sentence: str, demo_examples):
    """
    GPT-NER paper style few-shot prompt
    """
    prompt_lines = [
        "I am an excellent linguist.",
        "",
        "The task is to label person entities in the given Old English sentence using @@ and ##.",
        "If no person entity is present, return the sentence unchanged.",
        "",
        "Below are some examples.",
        ""
    ]

    for ex in demo_examples:
        demo_sentence = normalize_text(ex[sentence_col])
        demo_gold_names = parse_gold_names(ex[name_col])
        demo_output = mark_gold_names(demo_sentence, demo_gold_names)

        prompt_lines.append(f"Input: {demo_sentence}")
        prompt_lines.append(f"Output: {demo_output}")
        prompt_lines.append("")

    prompt_lines.append(f"Input: {test_sentence}")
    prompt_lines.append("Output:")

    return "\n".join(prompt_lines)

def call_gpt(prompt: str):
    response = client.responses.create(
        model=MODEL_NAME,
        input=prompt,
    )

    # SDK versions အလိုက် output_text access ကွာနိုင်လို့ safe extraction
    output_text = getattr(response, "output_text", None)
    if output_text is None:
        try:
            output_text = response.output[0].content[0].text
        except Exception:
            output_text = str(response)

    return output_text.strip()

def compute_metrics(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    return precision, recall, f1

def load_train_test_fold(fold_id):
    train_path = os.path.join(data_dir, f"train_fold_{fold_id}.xlsx")
    test_path = os.path.join(data_dir, f"test_fold_{fold_id}.xlsx")

    if not os.path.exists(train_path):
        raise FileNotFoundError(f"Train file not found: {train_path}")
    if not os.path.exists(test_path):
        raise FileNotFoundError(f"Test file not found: {test_path}")

    train_df = pd.read_excel(train_path)
    test_df = pd.read_excel(test_path)

    return train_df, test_df

def get_random_demo_examples(train_df, k, seed_value):
    """
    train fold ထဲက random retrieval
    """
    if len(train_df) == 0:
        return []

    sample_size = min(k, len(train_df))
    return train_df.sample(n=sample_size, random_state=seed_value).to_dict("records")

# =========================================================
# 5) RUN ALL K VALUES
# =========================================================
all_k_summary = []

for k in k_values:
    print("\n" + "=" * 60)
    print(f"Running Random Retrieval with k = {k}")
    print("=" * 60)

    k_output_dir = os.path.join(output_dir, f"k_{k}")
    os.makedirs(k_output_dir, exist_ok=True)

    fold_results = []
    all_tp, all_fp, all_fn = 0, 0, 0

    for fold_id in range(1, 6):
        try:
            train_df, test_df = load_train_test_fold(fold_id)
        except Exception as e:
            print(f"[Skip] Fold {fold_id}: {e}")
            continue

        # column name check
        if sentence_col not in train_df.columns or name_col not in train_df.columns:
            print(f"[Error] Train Fold {fold_id}: columns not found.")
            print("Available columns:", train_df.columns.tolist())
            continue

        if sentence_col not in test_df.columns or name_col not in test_df.columns:
            print(f"[Error] Test Fold {fold_id}: columns not found.")
            print("Available columns:", test_df.columns.tolist())
            continue

        tp, fp, fn = 0, 0, 0
        row_outputs = []

        print(f"\n========== Fold {fold_id} | k = {k} ==========")
        print(f"Rows: {len(test_df)}")

        for i, row in test_df.iterrows():
            sentence = normalize_text(row[sentence_col])
            gold_names = parse_gold_names(row[name_col])

            # row တစ်ခုချင်းစီအတွက် deterministic random retrieval
            seed_value = RANDOM_SEED + k * 1000 + fold_id * 100 + i
            demo_examples = get_random_demo_examples(train_df, k, seed_value)

            prompt = build_prompt(sentence, demo_examples)

            try:
                pred_output = call_gpt(prompt)
                pred_names = extract_marked_names(pred_output)
            except Exception as e:
                pred_output = f"[ERROR] {e}"
                pred_names = []

            gold_set = {normalize_entity(x) for x in gold_names}
            pred_set = {normalize_entity(x) for x in pred_names}

            row_tp = len(gold_set & pred_set)
            row_fp = len(pred_set - gold_set)
            row_fn = len(gold_set - pred_set)

            tp += row_tp
            fp += row_fp
            fn += row_fn

            row_outputs.append({
                "sentence": sentence,
                "gold_name_raw": row[name_col],
                "gold_names_parsed": "; ".join(gold_names),
                "model_output": pred_output,
                "pred_names_parsed": "; ".join(pred_names),
                "row_tp": row_tp,
                "row_fp": row_fp,
                "row_fn": row_fn,
            })

            time.sleep(0.2)  # rate limit soft protection

        precision, recall, f1 = compute_metrics(tp, fp, fn)

        print(f"TP={tp}, FP={fp}, FN={fn}")
        print(f"Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")

        fold_results.append({
            "k": k,
            "fold": fold_id,
            "rows": len(test_df),
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        })

        all_tp += tp
        all_fp += fp
        all_fn += fn

        # save detailed predictions
        detail_df = pd.DataFrame(row_outputs)
        detail_out = os.path.join(k_output_dir, f"predictions_fold_{fold_id}.xlsx")
        detail_df.to_excel(detail_out, index=False)

    # =========================================================
    # 6) SUMMARY FOR CURRENT K
    # =========================================================
    summary_df = pd.DataFrame(fold_results)

    if not summary_df.empty:
        macro_precision = summary_df["precision"].mean()
        macro_recall = summary_df["recall"].mean()
        macro_f1 = summary_df["f1"].mean()

        micro_precision, micro_recall, micro_f1 = compute_metrics(all_tp, all_fp, all_fn)

        print("\n========== Fold-wise Results ==========")
        print(summary_df.to_string(index=False))

        print(f"\n========== Average Across 5 Folds | k = {k} ==========")
        print(f"Macro Precision: {macro_precision:.4f}")
        print(f"Macro Recall   : {macro_recall:.4f}")
        print(f"Macro F1       : {macro_f1:.4f}")

        print("\n========== Overall Micro Score ==========")
        print(f"TP={all_tp}, FP={all_fp}, FN={all_fn}")
        print(f"Micro Precision: {micro_precision:.4f}")
        print(f"Micro Recall   : {micro_recall:.4f}")
        print(f"Micro F1       : {micro_f1:.4f}")

        summary_out = os.path.join(k_output_dir, "fold_summary.xlsx")
        summary_df.to_excel(summary_out, index=False)

        final_txt = os.path.join(k_output_dir, "final_scores.txt")
        with open(final_txt, "w", encoding="utf-8") as f:
            f.write(f"Random Retrieval Results | k = {k}\n\n")
            f.write("Fold-wise Results\n")
            f.write(summary_df.to_string(index=False))
            f.write("\n\nAverage Across 5 Folds\n")
            f.write(f"Macro Precision: {macro_precision:.4f}\n")
            f.write(f"Macro Recall   : {macro_recall:.4f}\n")
            f.write(f"Macro F1       : {macro_f1:.4f}\n\n")
            f.write("Overall Micro Score\n")
            f.write(f"TP={all_tp}, FP={all_fp}, FN={all_fn}\n")
            f.write(f"Micro Precision: {micro_precision:.4f}\n")
            f.write(f"Micro Recall   : {micro_recall:.4f}\n")
            f.write(f"Micro F1       : {micro_f1:.4f}\n")

        all_k_summary.append({
            "k": k,
            "macro_precision": macro_precision,
            "macro_recall": macro_recall,
            "macro_f1": macro_f1,
            "micro_tp": all_tp,
            "micro_fp": all_fp,
            "micro_fn": all_fn,
            "micro_precision": micro_precision,
            "micro_recall": micro_recall,
            "micro_f1": micro_f1,
        })

# =========================================================
# 7) FINAL SUMMARY FOR ALL K
# =========================================================
all_k_summary_df = pd.DataFrame(all_k_summary)

if not all_k_summary_df.empty:
    print("\n" + "=" * 60)
    print("FINAL SUMMARY FOR ALL K")
    print("=" * 60)
    print(all_k_summary_df.to_string(index=False))

    all_k_summary_out = os.path.join(output_dir, "all_k_summary.xlsx")
    all_k_summary_df.to_excel(all_k_summary_out, index=False)

    all_k_txt = os.path.join(output_dir, "all_k_summary.txt")
    with open(all_k_txt, "w", encoding="utf-8") as f:
        f.write("Final Summary for All K\n\n")
        f.write(all_k_summary_df.to_string(index=False))

    print(f"\nSaved results to: {output_dir}")
else:
    print("No results were generated.")


Running Random Retrieval with k = 4

========== Fold 1 | k = 4 ==========
Rows: 186
TP=224, FP=25, FN=48
Precision=0.8996, Recall=0.8235, F1=0.8599

========== Fold 2 | k = 4 ==========
Rows: 186
TP=207, FP=34, FN=41
Precision=0.8589, Recall=0.8347, F1=0.8466

========== Fold 3 | k = 4 ==========
Rows: 186
TP=188, FP=25, FN=43
Precision=0.8826, Recall=0.8139, F1=0.8468

========== Fold 4 | k = 4 ==========
Rows: 186
TP=208, FP=23, FN=40
Precision=0.9004, Recall=0.8387, F1=0.8685

========== Fold 5 | k = 4 ==========
Rows: 185
TP=214, FP=22, FN=36
Precision=0.9068, Recall=0.8560, F1=0.8807

========== Fold-wise Results ==========
 k  fold  rows  tp  fp  fn  precision   recall       f1
 4     1   186 224  25  48   0.899598 0.823529 0.859885
 4     2   186 207  34  41   0.858921 0.834677 0.846626
 4     3   186 188  25  43   0.882629 0.813853 0.846847
 4     4   186 208  23  40   0.900433 0.838710 0.868476
 4     5   185 214  22  36   0.906780 0.856000 0.880658

========== Average Across